# Predicting Food Insecurity Risk from Conflict Dynamics in South Sudan

**Author:** [Your name]
**Module:** Introduction to Machine Learning — Summative Project
**Mission alignment:** This project supports the goal of fostering enduring peace and sustainable development in South Sudan and Africa at large, by exploring whether patterns of armed conflict can help anticipate food security crises — a step toward earlier, more targeted humanitarian and peacebuilding response.

**GitHub repository:** [link here]
**Demo video:** [link here]
**Written report:** [link here]

---

## Problem Statement

[Write your problem statement here in your own words — see your report's introduction for the full version with citations.]

## Data Sources

- **ACLED** (Armed Conflict Location & Event Data Project) — event-level conflict data for South Sudan. Accessed [DATE]. https://acleddata.com
- **FEWS NET** (Famine Early Warning Systems Network) — IPC food security phase classifications for South Sudan. Accessed [DATE]. https://fews.net

Full data dictionary, filter settings, and known issues: see `../data/README.md`.

## Setup

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

# Reproducibility: set seeds for every source of randomness used below
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

import sys
sys.path.insert(0, "..")
from src.data_loading import load_acled, load_ipc
from src.preprocessing import (
    standardize_county_names,
    find_unmatched_counties,
    aggregate_acled_to_county_month,
    pivot_ipc_wide,
    merge_conflict_food_security,
)

## 1. Data Loading

In [ ]:
acled = load_acled("../data/raw/acled_south_sudan.csv")
ipc = load_ipc("../data/raw/fewsnet_ipc_south_sudan.csv")

print("ACLED:", acled.shape)
print("IPC:", ipc.shape)
acled.head()

In [ ]:
ipc.head()

_TODO: add your own markdown notes here on what each dataset contains, its grain (one row = one event vs. one row = one area-period-phase), and why each is relevant to the problem. Every code cell with output should have a short markdown interpretation near it — that's part of what's graded under Code Quality and Documentation.

## 2. Exploratory Data Analysis

In [ ]:
acled.set_index("event_date").resample("M").size().plot(
    figsize=(12, 4), title="ACLED Events per Month, South Sudan"
)
plt.ylabel("Event count")
plt.tight_layout()
plt.savefig("../reports/figures/events_per_month.png", dpi=150)
plt.show()

In [ ]:
acled["event_type"].value_counts().plot(
    kind="barh", figsize=(8, 4), title="ACLED Event Type Distribution, South Sudan"
)
plt.tight_layout()
plt.savefig("../reports/figures/event_type_distribution.png", dpi=150)
plt.show()

In [ ]:
# TODO: extend the EDA. Useful directions to cover:
#   - fatalities distribution (consider log scale, it's likely heavily right-skewed)
#   - geographic spread of events across admin1/admin2
#   - IPC phase distribution over time (is food insecurity trending up or down?)
#   - missingness visualization for population_best
#   - a first look at whether conflict-heavy counties and high-IPC-phase counties overlap
#
# For each plot, add a markdown cell underneath interpreting what you actually see,
# not just what the code does.

## 3. Preprocessing & Feature Engineering

Standardize county names across both datasets, then check what's left unmatched. The title-casing below fixes most ACLED/FEWS NET capitalization mismatches mechanically — what remains needs your judgment.

In [ ]:
acled = standardize_county_names(acled, "admin2")
ipc = standardize_county_names(ipc, "Area")

unmatched = find_unmatched_counties(acled, ipc)
print(f"{len(unmatched)} unmatched county names remain:")
sorted(unmatched)

**TODO:** review the list above. For each name, decide and document: is this a spelling/formatting difference you should fix in `COUNTY_NAME_FIXES` (in `src/preprocessing.py`), a genuinely distinct administrative unit one dataset doesn't have, or a non-geographic category (e.g., `Returnees`) that should be filtered out entirely rather than matched? Write your reasoning here, then update `COUNTY_NAME_FIXES` and re-run the cell above until you're satisfied.

In [ ]:
acled_monthly = aggregate_acled_to_county_month(acled)
ipc_wide = pivot_ipc_wide(ipc)

print("ACLED monthly:", acled_monthly.shape)
print("IPC wide:", ipc_wide.shape)
acled_monthly.head()

In [ ]:
# TODO: lag_months=1 is a starting default, not a justified choice.
# Test alternative lag windows (e.g., 2 or 3 months) as part of your required
# experiments, and justify your final choice in the report.
merged = merge_conflict_food_security(acled_monthly, ipc_wide, lag_months=1)
print("Merged shape:", merged.shape)
print("Rows with a matched conflict record:", (merged["event_count"] > 0).sum(), "/", len(merged))
merged.head()

In [ ]:
# TODO: define your target variable and justify the choice in your report. Two natural options:
#   (a) Classification — a binary or multi-class label derived from pct_phase3plus
#       (e.g., "Crisis+" if pct_phase3plus exceeds some threshold — check how IPC
#       itself defines actionable thresholds before picking your own)
#   (b) Regression — predict pct_phase3plus directly as a continuous value
#
# Document your reasoning for which framing you choose.

In [ ]:
# TODO: handle remaining missing values (e.g., population_best is ~21% missing
# in the raw ACLED data before aggregation), encode any categorical features
# you plan to use (e.g., Level 1 / state), and scale numeric features as
# appropriate for the models you'll train. Justify each decision.

In [ ]:
# TODO: chronological train/test split (NOT a random split).
# This is time-ordered data — train on an earlier period, test on the most
# recent period, so you're evaluating genuine forecasting ability rather
# than letting future information leak into training.

## 4. Classical Machine Learning Models

Implement at least one classical ML approach (e.g., Logistic Regression, Random Forest, Gradient Boosting, SVM) using scikit-learn. Per the rubric you need 7+ systematically varied, well-documented experiments across your classical ML and deep learning approaches combined — log every run in `../docs/experiment_log.md` as you go, with your reasoning for each change, not just the resulting metric.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# TODO: your classical ML implementation and experiments here

## 5. Deep Learning — Sequential API

Build an MLP using `tf.keras.Sequential` on your engineered tabular features, with a `tf.data.Dataset` input pipeline (required by the assignment). Document your architecture choices — number of layers, units, activation, regularization — and why you chose them, not just what they are.

In [ ]:
# TODO: build a tf.data.Dataset pipeline from your processed features, e.g.:
# train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(len(X_train), seed=SEED).batch(32)
# val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(32)

# TODO: define, compile, and train a Sequential model. Log this run in the experiment table.

## 6. Deep Learning — Functional API

Build a second deep learning architecture using the Functional API — for example, a multi-branch model that processes conflict-history features and static geographic/population features through separate input branches before merging. This satisfies the assignment's requirement to use both the Sequential and Functional APIs, and gives you a genuine architectural comparison point for your report's discussion section.

In [ ]:
# TODO: Functional API model, training, and logging.

## 7. Experiment Log

Maintain `../docs/experiment_log.md` as your primary experiment record (it's graded on its own as the Experiment/Result Table criterion). The cell below is optional scaffolding if you'd rather build the table programmatically and export it.

In [ ]:
experiment_log = pd.DataFrame(columns=[
    "experiment_id", "approach", "hyperparameters", "train_size", "test_size",
    "accuracy", "precision", "recall", "f1", "auc", "notes_and_insights",
])
experiment_log

## 8. Model Evaluation & Error Analysis

For your best-performing models from each approach:

- Plot learning curves (training vs. validation loss/accuracy) and diagnose over/underfitting with reference to the actual curve shapes you observe, not a generic statement.
- Plot and interpret confusion matrices — which phases are most confused, and why might that be (e.g., adjacent IPC phases being harder to distinguish than distant ones)?
- Plot ROC/AUC curves if your target is classification-based, and discuss precision-recall tradeoffs given any class imbalance you found in the EDA.
- Discuss bias-variance tradeoffs across your different model approaches.
- Critically reflect on dataset limitations: residual county-name matching uncertainty, ACLED's likely underreporting in remote/low-connectivity areas, the 2017+ time restriction on the food security labels, the low conflict-data match rate for many county-months.
- Propose specific, justified improvements for future iterations.

In [ ]:
# TODO: learning curve plots

In [ ]:
# TODO: confusion matrix plots and interpretation

In [ ]:
# TODO: ROC/AUC plots and interpretation

## 9. Conclusions

TODO: summarize your findings, tie back explicitly to your mission — does this analysis suggest conflict indicators provide meaningful early-warning signal for food insecurity in South Sudan? — and discuss real-world implications and limitations honestly.